## 


<table style="margin: 0; text-align: left; width:100%">
    <tr>
        <td style="width: 150px; height: 150px; vertical-align: middle;">
            <img src="../../assets/tools.png" width="150" height="150" style="display: block;" />
        </td>
        <td>
            <h2 style="color:#00bfff;">Our First Agent with the OpenAI Agents SDK and Ollama</h2>
            <h4 style="color:#00bf00;">© 2026 Dr Elena Kramer, Braude College, Israel.</h4>
            <span style="color:#00bfff;">In this lab, we take our first look at the OpenAI Agents SDK - you won't believe how lightweight it is. We build a simple agent, run it on an Ollama model, and watch its trace locally in Arize Phoenix. The documentation is clear and simple and well worth a look: <a href="https://openai.github.io/openai-agents-python/">https://openai.github.io/openai-agents-python/</a>

Set `OLLAMA_BASE_URL` in the project-root `.env` file. The model name is configured in the notebook.
            </span>
        </td>
    </tr>
</table>

## Sidenote - installing

The framework is called `openai-agents` on pypi.org (beware: `agents` is a different, older library).  
For local tracing we also need Phoenix and its instrumentation for the Agents SDK:

`uv add openai-agents arize-phoenix openinference-instrumentation-openai-agents`

In [ ]:
from dotenv import load_dotenv
import os
import time
import asyncio

from openai import AsyncOpenAI
from agents import Agent, Runner, trace, set_default_openai_client, set_default_openai_api

## Connecting the Agents SDK to Ollama

The SDK is asynchronous, so we give it an `AsyncOpenAI` client pointing at Ollama, and tell it to use the Chat Completions API.

In [ ]:
load_dotenv(override=True)
MODEL = "gpt-oss:120b"
# Always remember to do this!
OLLAMA_BASE_URL = os.getenv("OLLAMA_BASE_URL")
if not OLLAMA_BASE_URL:
    raise ValueError("Set OLLAMA_BASE_URL in the project-root .env file")

ollama_client = AsyncOpenAI(base_url=OLLAMA_BASE_URL, api_key="ollama")
set_default_openai_client(ollama_client, use_for_tracing=False)
set_default_openai_api("chat_completions")

## Local tracing with Phoenix

By default the SDK sends traces to the OpenAI dashboard. We use Ollama, so instead we send them to **Arize Phoenix**, an open-source tracing UI that runs on our own machine.

**Step 1** - start the Phoenix server in a terminal (and leave it running):

`uv run phoenix serve`

**Step 2** - run the cell below. `register` connects to Phoenix, and the instrumentor sends every Agents SDK trace there (instead of to OpenAI).

Then open http://localhost:6006

In [ ]:
from phoenix.otel import register
from openinference.instrumentation.openai_agents import OpenAIAgentsInstrumentor

tracer_provider = register(project_name="agents-lab1", endpoint="http://localhost:6006/v1/traces")
OpenAIAgentsInstrumentor().instrument(tracer_provider=tracer_provider)

## A word about `async` / `await`

`Runner.run` is a **coroutine** (`async def`). Calling it doesn't run it - we have to `await` it.  
While it waits for the model, Python can run other coroutines - so `asyncio.gather` can run several of them at the same time.

Jupyter already has an event loop, so we can write `await` directly in a cell. In a `.py` script use `asyncio.run(main())`.

In [ ]:
async def say_hello(name, delay):
    await asyncio.sleep(delay)          # pretend we are waiting for the LLM
    return f"Hello {name}"

start = time.perf_counter()
print(await say_hello("Alice", 1))
print(await say_hello("Bob", 1))
print(f"One after the other: {time.perf_counter() - start:.1f}s")

start = time.perf_counter()
print(await asyncio.gather(say_hello("Alice", 1), say_hello("Bob", 1)))
print(f"With gather: {time.perf_counter() - start:.1f}s")

## Step 1 - make an agent

An agent is just a name, instructions (the system prompt) and a model.

In [ ]:
agent = Agent(name="Jokester", instructions="You are a joke teller", model=MODEL)

## Step 2 - run it inside a trace

`Runner.run(agent, prompt)` is the Agent Loop. `with trace(...)` gives the trace a name.

In [ ]:
with trace("Telling a joke"):
    result = await Runner.run(agent, "Tell a joke about Autonomous AI Agents")
    print(result.final_output)

## Now go and look at the trace

http://localhost:6006 → project **agents-lab1** → "Telling a joke".  
Click the spans to see the agent, the LLM call to Ollama, the prompt, the answer and the token count.

## Step 3 - several agents at once

Three jokes concurrently with `asyncio.gather`, all inside one trace.

In [ ]:
topics = ["Python", "exams", "coffee"]

start = time.perf_counter()
with trace("Three jokes in parallel"):
    results = await asyncio.gather(
        *[Runner.run(agent, f"Tell one short joke about {topic}") for topic in topics]
    )
print(f"Took {time.perf_counter() - start:.1f}s\n")

for topic, r in zip(topics, results):
    print(f"{topic}: {r.final_output}\n")

## Look at the trace again

In Phoenix, open "Three jokes in parallel" - the three agent runs start at the same time.  
(Ollama handles a limited number of requests in parallel - see `OLLAMA_NUM_PARALLEL` - so the speed-up depends on your server.)

# Congratulations!

An Agent, the Runner (the Agent Loop), `async` / `await`, and local traces in Phoenix.  
All running on our own Ollama model, in just a few lines of code.